# Foundry IQ knowledge base with Work IQ


This notebook creates a Foundry IQ knowledge base that includes a Work IQ knowledge source that connects to Microsoft 365 (Outlook, Teams, Calendar, OneDrive). The KB queries it at runtime to surface emails, chats, and calendar events relevant to the user's question.
This notebook uses `InteractiveBrowserCredential` to sign in the current user and request a Work IQ user assertion for `api://<app-id>/access_as_user`. Azure AI Search exchanges that assertion through a federated credential and calls Work IQ on behalf of the signed-in user, honoring Microsoft 365 permissions.

In [ ]:
# EntraAppAuthentication needs azure-search-documents 12.1.0b2 or later. Install it if this kernel has an older version.
import importlib.metadata as md

from packaging.version import Version

try:
    installed = Version(md.version("azure-search-documents"))
except md.PackageNotFoundError:
    installed = Version("0")

if installed < Version("12.1.0b2"):
    %pip install -q "azure-search-documents==12.1.0b2"
    raise SystemExit("Installed azure-search-documents 12.1.0b2. Restart the kernel, then run the notebook from the top.")

print(f"azure-search-documents {installed} is ready")

## Step 1: Set up environment variables

Load the configuration for your Azure AI Search and Azure OpenAI resources. The `.env` file in the project folder has the values needed for this part.

In [1]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-08-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
WORK_IQ_SEARCH_ENTRA_APP_ID = os.environ["WORK_IQ_SEARCH_ENTRA_APP_ID"]
WORK_IQ_SEARCH_FEDERATED_CREDENTIAL_ID = os.environ["WORK_IQ_SEARCH_FEDERATED_CREDENTIAL_ID"]
WORK_IQ_SEARCH_ENTRA_TENANT_ID = os.getenv("WORK_IQ_SEARCH_ENTRA_TENANT_ID", AZURE_TENANT_ID)

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

Environment variables loaded


## Step 2: Login and acquire a Work IQ user assertion

In order for knowledge base retrieval to use your identity with Work IQ, you need a token for a signed-in user who has access to Microsoft 365 data in this tenant. The token targets the Work IQ Entra app that `infra/create-workiq-entra.py` created.

Run the cell below to sign in. If successful, it acquires a user assertion for `api://<app-id>/access_as_user` and saves it for use as `query_work_iq_source_authorization` during retrieval.

In [3]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(
    client_id=WORK_IQ_SEARCH_ENTRA_APP_ID,
    tenant_id=WORK_IQ_SEARCH_ENTRA_TENANT_ID,
)
user_assertion = user_credential.get_token(
    f"api://{WORK_IQ_SEARCH_ENTRA_APP_ID}/access_as_user"
).token

print("Acquired Work IQ user assertion for logged-in user")

Acquired Work IQ user assertion for logged-in user


## Step 3: Create three knowledge sources

For this knowledge base, you'll create three knowledge sources: the same two search-index sources used earlier, plus a Work IQ knowledge source.

* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `workiq-knowledge-source`: Points to Work IQ for workplace context

In [4]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    EntraAppAuthentication,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
    WorkIQKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential, api_version=AZURE_SEARCH_API_VERSION)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Contoso Work IQ knowledge source",
    work_iq_parameters=WorkIQKnowledgeSourceParameters(
        entra_app_authentication=EntraAppAuthentication(
            application_id=WORK_IQ_SEARCH_ENTRA_APP_ID,
            federated_credential_id=WORK_IQ_SEARCH_FEDERATED_CREDENTIAL_ID,
            tenant_id=WORK_IQ_SEARCH_ENTRA_TENANT_ID,
        )
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)
print("Knowledge sources created")


Knowledge sources created


## Step 4: Create the multi-source + Work IQ knowledge base

A knowledge base combines the knowledge sources with an LLM and instructions for how retrieval should behave.

For this notebook, the knowledge base uses an `outputMode` of `answerSynthesis` so the attached model can answer the question directly. It also uses a `retrievalReasoningEffort` of `low` so the model can help with query planning and source selection.

In [5]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-workiq-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining indexed company documents and Work IQ",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for workplace context (emails, chats, events, meetings, etc). Use the search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


Knowledge base created


## Step 5: Query the knowledge base

Ask a two-part question: one half needs your actual inbox context from Work IQ, the other needs company policy from the HR index.

* _"What are my recent emails about the Professional Claw Hammer stock issue at Seattle?"_ → should come from **Work IQ**
* _"Which Contoso job roles are responsible for inventory strategy and budget oversight?"_ → should come from `hrdocs`

> ⏳ Expect to wait 1-2 minutes for the response, as Work IQ increases retrieval latency. The retrieval request specifies `max_runtime_in_seconds` to increase the time the knowledge base waits for the results from the sources.

In [6]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential, api_version=AZURE_SEARCH_API_VERSION
)

question = ("Check my recent Teams chats for messages about the Professional Claw Hammer. "
            "Summarize what colleagues are saying and what actions have been requested. " 
            "Which Contoso job roles are responsible for inventory strategy and budget oversight?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_work_iq_source_authorization=user_assertion,
)
display(Markdown(result.response[0].content[0].text))


I found one relevant recent Teams chat thread about the **Professional Claw Hammer**. [ref_id:0]

Colleagues are saying Seattle is facing an ongoing stock-out for **Professional Claw Hammer 16oz (SKU: HTHM001600)**, with the item out of stock for **three days** and repeatedly requested by customers. [ref_id:0] Jordan Lee said a contractor needed **five units** and that this was the **third escalation of the week** for the same SKU, with concern the customer might take business elsewhere if it cannot be fulfilled soon. [ref_id:0] Marcus Chen said the hammer is a **strong seller** in a **high-traffic** Seattle store. [ref_id:0] Priya Sharma said the weekly inventory report showed **Seattle at zero stock**, while **Bellevue and Redmond** appeared to have available inventory. [ref_id:0]

The actions requested in the chat were to **check inventory availability across stores**, **verify current stock levels** especially for Bellevue and Redmond, **coordinate a store-to-store transfer** to Seattle if stock exists elsewhere, **check Tacoma and online inventory** for surplus, **escalate to procurement for an emergency reorder** if other stores are also low, and **follow up on transfer paperwork** if a transfer is approved. [ref_id:0]

For Contoso job roles, the chat explicitly identifies **Inventory Analyst** and **Regional Operations Manager** as the closest roles involved in inventory-related decisions. [ref_id:0] However, I did **not** find any record that explicitly says which Contoso role is formally responsible for **inventory strategy** or **budget oversight**. [ref_id:0]

### Review the activity log

For this activity log, you will see a "workIQ" step for the call made to Work IQ, along with "searchIndex" steps for each query sent to a search index.

In [7]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "inputTokens": 1425,
    "outputTokens": 62,
    "modelName": "gpt-5.4-mini",
    "elapsedMs": 1629
  },
  {
    "type": "workIQ",
    "id": 1,
    "knowledgeSourceName": "workiq-knowledge-source",
    "queryTime": "2026-07-28T10:36:05.8915256Z",
    "count": 1,
    "elapsedMs": 25944,
    "workIQArguments": {
      "search": "Check my recent Teams chats for messages about the Professional Claw Hammer. Summarize what colleagues are saying and what actions have been requested. Which Contoso job roles are responsible for inventory strategy and budget oversight?"
    }
  },
  {
    "type": "modelAnswerSynthesis",
    "id": 2,
    "inputTokens": 3445,
    "outputTokens": 355,
    "modelName": "gpt-5.4-mini",
    "elapsedMs": 2176
  },
  {
    "type": "agenticReasoning",
    "id": 3,
    "reasoningTokens": 1947,
    "retrievalReasoningEffort": {
      "kind": "low"
    }
  }
]


### Review the references

For the Work IQ knowledge source, the references have `type: "workIQ"`. The `sourceData` contains `extracts` with a synthesized answer from the Work IQ agent, and each reference contains `attributions` with URLs that link to any referenced emails, documents, etc.

In [8]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

[
  {
    "type": "workIQ",
    "id": "0",
    "activitySource": 1,
    "sourceData": {
      "extracts": [
        {
          "text": "I found **1 relevant Teams chat thread** about the **Professional Claw Hammer** in your recent chats. [1](https://teams.microsoft.com/l/message/19:55fb17c51cd046e1bd9288b2888e3402@thread.v2/1785220846858?context=%7B%22contextType%22:%22chat%22%7D#6b771a)\n\n## What colleagues are saying\n\nIn [Seattle Store Ops - Professional Claw Hammer](https://teams.microsoft.com/l/message/19:55fb17c51cd046e1bd9288b2888e3402@thread.v2/1785220846858?context=%7B%22contextType%22:%22chat%22%7D&EntityRepresentationId=5d4a26a5-5240-40c0-b23a-bbb118b18a2d), multiple colleagues report ongoing stock-out issues for the **Professional Claw Hammer 16oz (SKU: HTHM001600)** at the Seattle location. [1](https://teams.microsoft.com/l/message/19:55fb17c51cd046e1bd9288b2888e3402@thread.v2/1785220846858?context=%7B%22contextType%22:%22chat%22%7D#6b771a)\n\n- [Jordan Lee](https://www

#### 🔍 Source Hunt

Look at the references printed above. Can you identify:

1. Which knowledge source answered the **email thread summary** question?
2. Which knowledge source answered the **job roles** question?

## ✅ Mission complete

**What you built:**

* ✓ **Work IQ Knowledge Source**: A source that connects your real Microsoft 365 data to the knowledge base, so the KB can retrieve emails, Teams messages, and calendar context at query time.
* ✓ **3-source Knowledge Base**: A single KB spanning HR docs, health docs, and live M365 workplace data, with the agent routing each sub-question to the right source.
* ✓ **Hybrid inbox and policy retrieval**: Work IQ surfaced the email thread about the Seattle stock outage. The HR index answered the budget management question.